# Data Preprocessing

This notebook prepares the raw Tanzania Tourism dataset for feature engineering and model development. It focuses on data quality checks, missing-value analysis, categorical value standardization, and validation.

### Main Objectives

- Load the training and test datasets and understand their structure.
- Identify missing values, duplicate records, inconsistent values, and unusual numerical observations.
- Apply consistent, explainable cleaning rules to both datasets.
- Validate the processed data before moving to feature engineering.

> **Note:** The target column is available only in the training dataset. The test dataset must keep its original rows and must not be used to fit preprocessing decisions.


## 1. Import Required Libraries

The following libraries and project modules are used in this notebook:

- `Path` and `sys` to locate the project root and import local modules.
- `NumPy` for numerical operations.
- `Pandas` for loading, inspecting, and cleaning tabular data.
- Project configuration variables for dataset paths and feature definitions.
- Utility functions for missing-value and outlier analysis.


In [1]:
from pathlib import Path
import sys 

notebook_dir = Path.cwd()
project_root = notebook_dir.parent 

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import TRAIN_DATA_PATH, TEST_DATA_PATH, VARIABLE_DEFINITIONS_PATH, NUMERIC_COLS, ID_COL, TARGET_COL, TARGET_CLASSES, CATEGORICAL_COLS, BASE_CATEGORICAL_COLS, BASE_NUMERIC_COLS
from src.utils import outliers, massing_analysis
import numpy as np
import pandas as pd



## 2. Load Raw Datasets

Load the training data, test data, and variable definitions. The variable definitions file helps verify the meaning of each feature before making cleaning decisions.


### Load the Source Files

Read the CSV files using the paths defined in the project configuration. At this stage, the data is loaded without applying cleaning transformations.


In [2]:
train_df = pd.read_csv(TRAIN_DATA_PATH)
test_df = pd.read_csv(TEST_DATA_PATH)
vb_df = pd.read_csv(VARIABLE_DEFINITIONS_PATH)


The three source files are now loaded into separate DataFrames. The training and test datasets will be inspected independently, while the variable definitions are used as documentation.


### Preview the Training Data

Display a small sample to inspect column names, example values, and the general layout of the labelled dataset.


In [3]:
train_df.head(5)

,Tour_ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,...,package_accomodation,package_food,package_transport_tz,package_sightseeing,package_guided_tour,package_insurance,night_mainland,night_zanzibar,first_trip_tz,cost_category
0,tour_id1hffseyw,ITALY,45-64,With Children,0.0,2.0,Visiting Friends and Relatives,Beach Tourism,"Friends, relatives",Package Tour,...,Yes,Yes,Yes,No,No,No,0,7,Yes,High Cost
1,tour_idnacd7zag,UNITED KINGDOM,25-44,With Spouse,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,No,No,No,0,7,Yes,High Cost
2,tour_id62vz7e71,UNITED STATES OF AMERICA,65+,With Spouse,1.0,1.0,Leisure and Holidays,Widlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,Yes,Yes,No,6,6,Yes,Higher Cost
3,tour_idrc76tzix,RWANDA,25-44,With Spouse and Children,3.0,1.0,Leisure and Holidays,Beach Tourism,"Radio, TV, Web",Independent,...,No,No,No,No,No,No,3,0,No,Lower Cost
4,tour_idn723m0n9,UNITED STATES OF AMERICA,45-64,Alone,0.0,1.0,Leisure and Holidays,Widlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,No,Yes,Yes,7,0,Yes,Higher Cost


In [4]:
train_df.shape


(18506, 21)

### Preview the Test Data

Inspect the test features and confirm that the target column is not included. The test set should contain the same predictor columns as the training set.


In [5]:
test_df.head(5)

,Tour_ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,package_transport_int,package_accomodation,package_food,package_transport_tz,package_sightseeing,package_guided_tour,package_insurance,night_mainland,night_zanzibar,first_trip_tz
0,tour_idynufedne,KOREA,25-44,Alone,0.0,1.0,Leisure and Holidays,Widlife Tourism,Others,Independent,No,No,No,No,No,No,No,7,4,Yes
1,tour_id9r3y5moe,UNITED KINGDOM,45-64,With Children,1.0,1.0,Leisure and Holidays,Conference Tourism,"Travel agent, tour operator",Package Tour,Yes,Yes,Yes,Yes,Yes,Yes,Yes,7,0,Yes
2,tour_idf6itml6g,ITALY,25-44,With Spouse,1.0,1.0,Leisure and Holidays,Beach Tourism,"Travel agent, tour operator",Package Tour,Yes,Yes,No,Yes,No,No,No,0,6,Yes
3,tour_id99u4znru,KENYA,25-44,Alone,0.0,1.0,Other,Beach Tourism,"Radio, TV, Web",Independent,No,No,No,No,No,No,No,3,4,No
4,tour_idj4i9urbx,ZAMBIA,25-44,Alone,0.0,1.0,Business,Widlife Tourism,"Radio, TV, Web",Independent,No,No,No,No,No,No,No,6,0,No


In [6]:
test_df.shape


(6169, 20)

### Review the Variable Definitions

Use the feature descriptions as a reference when interpreting columns and deciding whether a value is invalid or simply uncommon.


In [7]:
vb_df

,Column Name,Definition
0,id,Unique identifier for each tourist
1,country,The country a tourist coming from.
2,age_group,The age group of a tourist.
3,travel_with,The relation of people a tourist travel with t...
4,total_female,Total number of females
5,total_male,Total number of males
6,purpose,The purpose of visiting Tanzania
7,main_activity,The main activity of tourism in Tanzania
8,infor_source,The source of information about tourism in Tan...
9,tour_arrangment,The arrangment of visiting Tanzania


The definitions file describes the intended meaning of the columns. Some names in the file may differ slightly from the actual dataset column names, so use the DataFrame schema as the source of truth when writing code.


## 3. Inspect Data Quality

Start with a general audit of both datasets before changing any values. This helps identify issues that may affect later preprocessing and model performance.

The audit covers:

- Dataset structure and data types
- Missing values
- Duplicate records
- Numerical distributions and potential outliers
- Invalid or inconsistent values


### Inspect Dataset Structure

Check the number of records, non-null values, data types, and memory usage in both datasets. This provides a baseline before any cleaning operations.


In [8]:
print('train dataframe')
display(train_df.info())
print('_'*50,'\n') 
print('test dataframe')
display(test_df.info())

train dataframe
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18506 entries, 0 to 18505
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Tour_ID                18506 non-null  object 
 1   country                18506 non-null  object 
 2   age_group              18506 non-null  object 
 3   travel_with            17431 non-null  object 
 4   total_female           18504 non-null  float64
 5   total_male             18500 non-null  float64
 6   purpose                18506 non-null  object 
 7   main_activity          18506 non-null  object 
 8   info_source            18506 non-null  object 
 9   tour_arrangement       18506 non-null  object 
 10  package_transport_int  18506 non-null  object 
 11  package_accomodation   18506 non-null  object 
 12  package_food           18506 non-null  object 
 13  package_transport_tz   18506 non-null  object 
 14  package_sightseeing    18506 non-null 

None

__________________________________________________ 

test dataframe
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6169 entries, 0 to 6168
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Tour_ID                6169 non-null   object 
 1   country                6169 non-null   object 
 2   age_group              6169 non-null   object 
 3   travel_with            5808 non-null   object 
 4   total_female           6167 non-null   float64
 5   total_male             6168 non-null   float64
 6   purpose                6169 non-null   object 
 7   main_activity          6169 non-null   object 
 8   info_source            6169 non-null   object 
 9   tour_arrangement       6169 non-null   object 
 10  package_transport_int  6169 non-null   object 
 11  package_accomodation   6169 non-null   object 
 12  package_food           6169 non-null   object 
 13  package_transport_tz   6169 non-null   o

None

The training dataset contains **18,506 rows and 21 columns**, while the test dataset contains **6,169 rows and 20 columns**. The target column is present only in train. Most features are stored as `object`, and the numerical columns include integer and floating-point types.


### Measure Missing Values

Count missing values in each column and calculate their percentage. This identifies which features need attention and helps prioritize the cleaning work.


In [9]:
print("Missing Values in train dataset : ")
display(massing_analysis(train_df))
print('_'*50,'\n') 
print("Missing Values in test dataset : ")
display(massing_analysis(test_df))


Missing Values in train dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,1075.0,5.81
total_female,2.0,0.01
total_male,6.0,0.03
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


__________________________________________________ 

Missing Values in test dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,361.0,5.85
total_female,2.0,0.03
total_male,1.0,0.02
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


The main issue is missing data in `travel_with`: **1,075 values (5.81%)** in train and **361 values (5.85%)** in test. A small number of values are also missing in `total_female` and `total_male`. These features are reviewed together because they describe the travel group.


### Check Fully Duplicated Rows

Check for exact duplicate rows in each dataset. This is different from checking whether the ID column contains duplicates, which is validated later.


In [10]:
print("Duplicate Records in train dataset : ",train_df.duplicated().sum())
print("Duplicate Records in test dataset : ",test_df.duplicated().sum())


Duplicate Records in train dataset :  0
Duplicate Records in test dataset :  0


No exact duplicate rows were found in either dataset. The ID column is checked separately later to confirm identifier uniqueness.


### Detect Potential Outliers

Use the project's outlier utility to flag unusual observations in the numerical columns. Treat these results as candidates for investigation, not as rows that must be removed.


In [11]:
print("outliers for train dataset ")
outliers(train_df)
print('_'*50,'\n') 
print("outliers for test dataset ")
outliers(test_df)


outliers for train dataset 
total_female | outliers : 946 - 5.11%
total_male | outliers : 6692 - 36.16%
night_mainland | outliers : 1148 - 6.20%
night_zanzibar | outliers : 729 - 3.94%
__________________________________________________ 

outliers for test dataset 
total_female | outliers : 320 - 5.19%
total_male | outliers : 2204 - 35.73%
night_mainland | outliers : 383 - 6.21%
night_zanzibar | outliers : 271 - 4.39%


The outlier utility flags observations in all four numerical features. `total_male` has the largest flagged proportion. These results suggest skewed or extreme values, but they do not establish that the observations are incorrect.


### Review Numerical Distributions

Use descriptive statistics to inspect central values, spread, minimums, and maximums for the numerical features.


In [13]:
train_df[BASE_NUMERIC_COLS].describe()

,total_female,total_male,night_mainland,night_zanzibar
count,18504.000000,18500.000000,18506.000000,18506.000000
mean,0.936230,0.998757,9.141576,2.493516
std,1.215582,1.173177,14.127449,5.275156
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,1.000000,3.000000,0.000000
50%,1.000000,1.000000,6.000000,0.000000
75%,1.000000,1.000000,11.000000,4.000000
max,49.000000,58.000000,365.000000,240.000000


The numerical features have non-negative minimum values. Their maximum values vary widely, so the distributions require further review before deciding whether any observations should be capped or removed.


**Initial numerical check:** All four numerical features have a minimum value of zero, so no negative values are visible in this summary. This does not prove that every value is valid; maximum values and domain constraints are reviewed separately.


## 4. Handle Missing Values

Investigate missing values before choosing an imputation strategy. Use domain information only when it supports a reasonable conclusion, and keep the same cleaning logic for training and test data.

The first focus is `travel_with`, which has the largest number of missing values.


### Create a Reusable Missing-Value Report

Define a small helper function so the same missing-value analysis can be run after each cleaning step. Reusing one function makes before-and-after comparisons easier.


In [14]:
# fast missing values analysis 

def fmva(train_df, test_df=None):
    print("Missing Values in train dataset : ")
    display(massing_analysis(train_df))
    if test_df is not None:
        print('_'*50,'\n') 
        print("Missing Values in test dataset : ")
        display(massing_analysis(test_df))


### Establish the Initial Missing-Value Baseline

Run the helper function before changing the data. Keep these counts as the baseline for evaluating the effect of each imputation rule.


In [15]:
fmva(train_df, test_df)

Missing Values in train dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,1075.0,5.81
total_female,2.0,0.01
total_male,6.0,0.03
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


__________________________________________________ 

Missing Values in test dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,361.0,5.85
total_female,2.0,0.03
total_male,1.0,0.02
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


### Inspect Existing `travel_with` Categories

Review the available values to understand the valid categories and confirm that missing values are present.


In [16]:
train_df['travel_with'].unique()

array(['With Children', 'With Spouse', 'With Spouse and Children',
       'Alone', nan, 'With Other Friends/Relatives'], dtype=object)

### Infer Travellers Recorded as `Alone`

A missing `travel_with` value is set to `Alone` when the recorded group contains no more than one person (`total_female + total_male <= 1`). This is a domain-based rule and is applied identically to train and test.

Rows with missing group counts are not classified by this rule because the condition cannot be evaluated reliably.


In [17]:
def fill_Alone_values(df):
    mask_alone = (df['total_female'] + df['total_male']) <= 1 
    mask_nan = df['travel_with'].isna()
    mask = mask_nan & mask_alone
    df.loc[mask, 'travel_with'] = 'Alone'

fill_Alone_values(train_df)
fill_Alone_values(test_df)

This rule updates only rows that meet the stated condition. It does not replace missing group counts and does not infer a category when the available information is insufficient.


### Recheck Missing Values After the First Rule

Compare the new missing-value counts with the baseline. This shows how many missing `travel_with` values were resolved by the `Alone` rule and what still needs investigation.


In [18]:
fmva(train_df, test_df)

Missing Values in train dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,28.0,0.15
total_female,2.0,0.01
total_male,6.0,0.03
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


__________________________________________________ 

Missing Values in test dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,7.0,0.11
total_female,2.0,0.03
total_male,1.0,0.02
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


The first rule reduces missing `travel_with` values substantially. Remaining records need a separate review rather than being assigned a category without supporting evidence.


### Investigate Remaining Two-Person Groups

Find records where `travel_with` is missing but the group contains exactly one female and one male. Review these records in both datasets before deciding whether a spouse label is justified.


In [19]:
# one male and female
def omf(df):
    mask_nan = df['travel_with'].isna()
    mask_female = df['total_female'] ==1 
    mask_male   = df['total_male'] == 1 

    omale_and_ofemale_train = df[mask_nan & mask_female & mask_male]
    return omale_and_ofemale_train

print('train dataframe :')
display(omf(train_df))
print('test dataframe :')
display(omf(test_df))


train dataframe :


,Tour_ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,...,package_accomodation,package_food,package_transport_tz,package_sightseeing,package_guided_tour,package_insurance,night_mainland,night_zanzibar,first_trip_tz,cost_category
5926,tour_idrc9ck93b,ITALY,25-44,NaN,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,Yes,Yes,No,9,0,Yes,Higher Cost
8745,tour_idqgl7rdqa,UNITED STATES OF AMERICA,25-44,NaN,1.0,1.0,Leisure and Holidays,Conference Tourism,"Radio, TV, Web",Independent,...,No,No,No,No,No,No,11,0,Yes,High Cost
13499,tour_idivz7lpc8,ITALY,25-44,NaN,1.0,1.0,Leisure and Holidays,Beach Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,No,No,No,0,7,Yes,Normal Cost
15213,tour_idb9xsmbi1,UNITED KINGDOM,25-44,NaN,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,No,Yes,Yes,No,7,0,Yes,Normal Cost
15994,tour_idp90whjre,ITALY,25-44,NaN,1.0,1.0,Leisure and Holidays,Beach Tourism,"Friends, relatives",Package Tour,...,Yes,Yes,Yes,Yes,Yes,Yes,0,7,Yes,Higher Cost
16030,tour_ide70fvj1y,ITALY,45-64,NaN,1.0,1.0,Leisure and Holidays,Beach Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,Yes,Yes,Yes,11,6,No,Highest Cost
18070,tour_idc45a73cc,SWEDEN,18-24,NaN,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,No,No,No,No,No,No,32,7,Yes,Normal Cost


test dataframe :


,Tour_ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,package_transport_int,package_accomodation,package_food,package_transport_tz,package_sightseeing,package_guided_tour,package_insurance,night_mainland,night_zanzibar,first_trip_tz
953,tour_idd0r31y09,EGYPT,25-44,NaN,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Newspaper, magazines, brochures",Independent,No,No,No,No,No,No,No,23,3,Yes
2096,tour_idobv4yb8i,ITALY,45-64,NaN,1.0,1.0,Leisure and Holidays,Beach Tourism,"Travel agent, tour operator",Package Tour,Yes,Yes,Yes,No,No,No,No,0,7,Yes
2721,tour_idqi6nvow6,CANADA,45-64,NaN,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,No,Yes,Yes,Yes,Yes,Yes,No,5,0,No
2776,tour_id0ue6iywk,ITALY,25-44,NaN,1.0,1.0,Leisure and Holidays,Cultural Tourism,"Radio, TV, Web",Package Tour,Yes,Yes,Yes,Yes,Yes,Yes,Yes,7,7,Yes


The output contains **7 matching train records and 4 matching test records**. All displayed records have the purpose `Leisure and Holidays`, which supports further investigation but should still be treated as an inference rather than a guaranteed fact.


### Compare with Known Spouse Records

Use records already labelled `With Spouse` and containing one female and one male to examine their purpose distribution. This gives supporting evidence, but it does not prove that every similar record represents a spouse.


In [20]:
mask_nan = train_df['travel_with'] == 'With Spouse'
mask_female = train_df['total_female'] == 1 
mask_male   = train_df['total_male'] == 1 

travel_with_spouse = train_df[mask_nan & mask_female & mask_male]
travel_with_spouse.purpose.value_counts()

purpose
Leisure and Holidays              4213
Visiting Friends and Relatives     213
Business                            75
Meetings and Conference             35
Other                               19
Volunteering                        19
Scientific and Academic              5
Medical                              2
Name: count, dtype: int64

Among known spouse records with one female and one male, `Leisure and Holidays` is the most common purpose. However, other purposes are also present, including `Visiting Friends and Relatives`; therefore, the inference rule should combine more than one condition.


The most common purpose among records known to be travelling with a spouse is **Leisure and Holidays**. However, other purposes also occur, so purpose alone should not be used to infer the relationship. The rule below combines purpose with the group-size information.


### Apply the Spouse Inference Rule

Set `travel_with` to `With Spouse` only when the value is missing, the group contains one female and one male, and the purpose is either `Leisure and Holidays` or `Visiting Friends and Relatives`. The same rule is applied to both datasets.


In [21]:
# with Spouse handel data

def wshd(df):
    mask_nan = df['travel_with'].isna()
    mask_female = df['total_female'] == 1  
    mask_male   = df['total_male'] == 1 
    mask_purpose = df['purpose'].isin(['Leisure and Holidays', 'Visiting Friends and Relatives'])

    mask = mask_nan & mask_female & mask_male & mask_purpose
    df.loc[mask, 'travel_with'] = 'With Spouse' 

wshd(train_df)
wshd(test_df)


The spouse rule resolves additional missing values. The next report shows that some values remain in both datasets, including missing numerical group counts.


### Evaluate the Updated Missing Values

Run the report again to measure the remaining missing values after both inference rules. Do not assume that all missing data has been resolved.


In [22]:
fmva(train_df, test_df)


Missing Values in train dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,21.0,0.11
total_female,2.0,0.01
total_male,6.0,0.03
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


__________________________________________________ 

Missing Values in test dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,3.0,0.05
total_female,2.0,0.03
total_male,1.0,0.02
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


After the two inference rules, `travel_with` still has **21 missing values in train** and **3 in test**. The training data also has 2 missing `total_female` values and 6 missing `total_male` values; test has 2 and 1 respectively.


### Remove Incomplete Training Rows

The current code drops every training row that still contains at least one missing value. This removes 28 rows from the original 18,506 training records.

**Methodological note:** Row deletion is a choice, not a neutral cleaning step. Before keeping it, compare it with imputing the remaining numerical and categorical values. For a production pipeline, fit imputers on training data only and apply the fitted imputers to validation and test data. Never drop test rows.


In [23]:
train_df = train_df.dropna()


### Confirm the Training Data After Row Removal

Check that the training dataset now contains 18,478 rows and that all columns are non-null. This result follows from the `dropna()` operation above.


In [24]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18478 entries, 0 to 18505
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Tour_ID                18478 non-null  object 
 1   country                18478 non-null  object 
 2   age_group              18478 non-null  object 
 3   travel_with            18478 non-null  object 
 4   total_female           18478 non-null  float64
 5   total_male             18478 non-null  float64
 6   purpose                18478 non-null  object 
 7   main_activity          18478 non-null  object 
 8   info_source            18478 non-null  object 
 9   tour_arrangement       18478 non-null  object 
 10  package_transport_int  18478 non-null  object 
 11  package_accomodation   18478 non-null  object 
 12  package_food           18478 non-null  object 
 13  package_transport_tz   18478 non-null  object 
 14  package_sightseeing    18478 non-null  object 
 15  package

The resulting training dataset contains **18,478 complete rows**. The reduction from 18,506 rows is caused by dropping rows with at least one remaining missing value. Keep this trade-off documented because it affects the training sample.


### Verify Missing Values in the Cleaned Training Set

Run the missing-value report on the training data after the row-removal step. The expected result is zero missing values in every training column.


In [28]:
fmva(train_df)

Missing Values in train dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,,
total_female,,
total_male,,
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


The output shows no missing values in the training dataset after the `dropna()` operation. This does not mean the test dataset is complete.


## 5. Check Duplicate Records

Check whether either dataset contains fully duplicated rows. This step is a diagnostic check; remove duplicates only when they are confirmed to be redundant.


### Report Duplicate-Row Counts

The output indicates zero fully duplicated rows in both datasets. No row removal is required based on this check.


In [29]:
print('Number of duplicate records in train dataset:', train_df.duplicated().sum())
print('Number of duplicate records in test dataset:', test_df.duplicated().sum())

Number of duplicate records in train dataset: 0
Number of duplicate records in test dataset: 0


Both datasets have **zero fully duplicated rows**. The duplicate check does not require any further action.


## 6. Validate Data Types and Schema

Compare the data types of corresponding columns in the training and test datasets. The goal is to confirm that shared features have compatible types and that the target column exists only in the training dataset.


### Inspect Training Data Types

Review the data types assigned to every training column. Object columns are expected for IDs and categorical variables, while numerical measurements should remain numeric.


In [30]:
train_df.dtypes


Tour_ID                   object
country                   object
age_group                 object
travel_with               object
total_female             float64
total_male               float64
purpose                   object
main_activity             object
info_source               object
tour_arrangement          object
package_transport_int     object
package_accomodation      object
package_food              object
package_transport_tz      object
package_sightseeing       object
package_guided_tour       object
package_insurance         object
night_mainland             int64
night_zanzibar             int64
first_trip_tz             object
cost_category             object
dtype: object

### Inspect Test Data Types

Review the test data types and compare them with the corresponding training features. The target column is expected to be absent from test.


In [31]:
test_df.dtypes

Tour_ID                   object
country                   object
age_group                 object
travel_with               object
total_female             float64
total_male               float64
purpose                   object
main_activity             object
info_source               object
tour_arrangement          object
package_transport_int     object
package_accomodation      object
package_food              object
package_transport_tz      object
package_sightseeing       object
package_guided_tour       object
package_insurance         object
night_mainland             int64
night_zanzibar             int64
first_trip_tz             object
dtype: object

### Compare Shared Column Types

Build a side-by-side comparison of training and test data types. This helps identify inconsistent representations before encoding or model fitting.


In [32]:

data_values_com = pd.DataFrame({
    'train types' : train_df.dtypes,
    'test types' : test_df.dtypes})


### Review the Schema Comparison

Confirm that shared feature columns have compatible types. The missing `cost_category` entry in test is expected because it is the target variable.


In [33]:

data_values_com

,train types,test types
Tour_ID,object,object
age_group,object,object
cost_category,object,NaN
country,object,object
first_trip_tz,object,object
info_source,object,object
main_activity,object,object
night_mainland,int64,int64
night_zanzibar,int64,int64
package_accomodation,object,object


The shared columns use matching data types. `cost_category` appears only in train, as expected for the target variable.


### Check Numerical Feature Ranges

Review descriptive statistics for the numerical features and confirm that their observed ranges are reasonable. In particular, check minimum values for impossible negative counts.


### Compare Numerical Feature Statistics

Review the numerical summaries for both datasets. Check minimum values, distribution differences, and unusually large maximum values.


In [35]:
display(train_df[BASE_NUMERIC_COLS].describe().T)
display(test_df[BASE_NUMERIC_COLS].describe().T)

,count,mean,std,min,25%,50%,75%,max
total_female,18478.0,0.932839,1.204878,0.0,0.0,1.0,1.0,49.0
total_male,18478.0,0.997727,1.172420,0.0,1.0,1.0,1.0,58.0
night_mainland,18478.0,9.140491,14.132483,0.0,3.0,6.0,11.0,365.0
night_zanzibar,18478.0,2.495671,5.278308,0.0,0.0,0.0,4.0,240.0


,count,mean,std,min,25%,50%,75%,max
total_female,6167.0,0.922491,1.173067,0.0,0.0,1.0,1.0,30.0
total_male,6168.0,1.017510,1.526274,0.0,1.0,1.0,1.0,90.0
night_mainland,6169.0,9.311720,16.366900,0.0,3.0,6.0,11.0,664.0
night_zanzibar,6169.0,2.585832,5.465058,0.0,0.0,0.0,4.0,174.0


The numerical features have non-negative minimum values in both datasets. Several maximum values are much larger than their medians, which indicates strongly right-skewed distributions.


### Inspect Values and Underlying Python Types

Loop through the training columns to review their unique values, pandas data types, and underlying Python value types. This is useful for detecting mixed-type columns.


In [36]:
for col in train_df.columns:
    print(train_df[col].name,'\n', train_df[col].unique())
    print(train_df[col].dtype)
    print(train_df[col].map(type).value_counts(),'\n')


Tour_ID 
 ['tour_id1hffseyw' 'tour_idnacd7zag' 'tour_id62vz7e71' ...
 'tour_idv7pz3vs8' 'tour_idy6ydo00w' 'tour_idceoq9por']
object
Tour_ID
<class 'str'>    18478
Name: count, dtype: int64 

country 
 ['ITALY' 'UNITED KINGDOM' 'UNITED STATES OF AMERICA' 'RWANDA' 'FRANCE'
 'SWIZERLAND' 'NIGERIA' 'AUSTRALIA' 'SPAIN' 'DENMARK' 'BELGIUM' 'ZAMBIA'
 'NETHERLANDS' 'CANADA' 'INDIA' 'NORWAY' 'ZIMBABWE' 'MORROCO' 'CHINA'
 'GERMANY' 'SOUTH AFRICA' 'JAPAN' 'UAE' 'RUSSIA' 'KENYA' 'GREECE' 'POLAND'
 'AUSTRIA' 'MALAYSIA' 'CZECH REPUBLIC' 'KOREA' 'DRC' 'CYPRUS' 'KUWAIT'
 'BRAZIL' 'CHILE' 'BULGARIA' 'UNITED ARAB EMIRATES' 'ISRAEL' 'NEW ZEALAND'
 'UGANDA' 'IRELAND' 'SWEDEN' 'QATAR' 'CONGO' 'BURUNDI' 'FINLAND' 'GHANA'
 'OMAN' 'LEBANON' 'LUXEMBOURG' 'IVORY COAST' 'NAMIBIA' 'ROMANIA'
 'THAILAND' 'SUDAN' 'SINGAPORE' 'PORTUGAL' 'MALAWI' 'MALT' 'EGYPT'
 'PAKISTAN' 'TANZANIA' 'ARGENTINA' 'SRI LANKA' 'BOTSWANA' 'VIETNAM'
 'BERMUDA' 'TAIWAN' 'SENEGAL' 'AFGHANISTAN' 'COLOMBIA' 'JAMAICA' 'BAHRAIN'
 'COSTARICA' 'CO

The `travel_with` column contains both strings and floating-point values in the training dataset. The next checks determine whether those floating-point entries are missing values represented as `NaN`, rather than valid numeric categories.


### Count Missing `travel_with` Values in Training

Check the exact number of missing values remaining in the training column before interpreting the type inspection.


In [37]:
print(train_df["travel_with"].isna().sum())


0


### Identify Non-String, Non-Missing Values

Filter for values that are neither strings nor missing. An empty result means there are no unexpected values of another type in this column.


In [38]:

mask = ~train_df['travel_with'].map(lambda value: 
            isinstance(value, str) or pd.isna(value))
train_df.loc[mask, "travel_with"]

Series([], Name: travel_with, dtype: object)

The result is empty, confirming that all non-missing values in the training `travel_with` column are strings.


The check confirms that the non-string entries in `travel_with` are missing values. No unexpected numeric category values were found in this column.


### Inspect Values and Types in the Test Dataset

Repeat the same type inspection for test to make sure that the shared feature is represented consistently.


In [39]:
for col in test_df.columns:
    print(test_df[col].name,'\n', test_df[col].unique())
    print(test_df[col].dtype)
    print(test_df[col].map(type).value_counts(),'\n')


Tour_ID 
 ['tour_idynufedne' 'tour_id9r3y5moe' 'tour_idf6itml6g' ...
 'tour_id7wwqrs0p' 'tour_idx80vbw5a' 'tour_id8fkkwytb']
object
Tour_ID
<class 'str'>    6169
Name: count, dtype: int64 

country 
 ['KOREA' 'UNITED KINGDOM' 'ITALY' 'KENYA' 'ZAMBIA' 'UGANDA' 'NETHERLANDS'
 'OMAN' 'ZIMBABWE' 'BELGIUM' 'DRC' 'UNITED STATES OF AMERICA' 'FRANCE'
 'CANADA' 'CZECH REPUBLIC' 'UAE' 'SPAIN' 'SOUTH AFRICA' 'SWEDEN' 'NORWAY'
 'CHINA' 'GERMANY' 'RWANDA' 'INDIA' 'AUSTRIA' 'DJIBOUT' 'IRELAND'
 'SWIZERLAND' 'AUSTRALIA' 'PHILIPINES' 'BURUNDI' 'JAPAN' 'THAILAND'
 'PORTUGAL' 'DENMARK' 'MOZAMBIQUE' 'UNITED ARAB EMIRATES' 'TAIWAN'
 'BRAZIL' 'HUNGARY' 'SRI LANKA' 'TUNISIA' 'INDONESIA' 'ISRAEL'
 'NEW ZEALAND' 'POLAND' 'CONGO' 'ESTONIA' 'COLOMBIA' 'MALAYSIA' 'TANZANIA'
 'BULGARIA' 'RUSSIA' 'COMORO' 'UKRAIN' 'LEBANON' 'URUGUAY' 'SOMALI'
 'FINLAND' 'ETHIOPIA' 'SCOTLAND' 'TURKEY' 'SLOVENIA' 'MORROCO' 'SWAZILAND'
 'LUXEMBOURG' 'MALAWI' 'SINGAPORE' 'SUDAN' 'PAPUA NEW GUINEA' 'PAKISTAN'
 'EGYPT' 'CYPRUS' 'QATAR' 

The same type check is required for the test dataset to make sure that its `travel_with` column follows the same pattern.


### Count Missing `travel_with` Values in Test

Check the exact number of missing values in the test column.


In [40]:
print(test_df["travel_with"].isna().sum())


3


### Identify Unexpected Test Values

Use the same non-string, non-missing filter on test. An empty result confirms that the non-string entries are missing values rather than unexpected category values.


In [41]:
mask = ~test_df['travel_with'].map(lambda value: 
            isinstance(value, str) or pd.isna(value))
test_df.loc[mask, "travel_with"]

Series([], Name: travel_with, dtype: object)

The result is empty, confirming that the non-string entries in the test `travel_with` column are missing values.


The floating-point entries in the test `travel_with` column are also missing values, not valid category values.


### Validate the Identifier Column

Check that `Tour_ID` is stored as text in both datasets and that its underlying values are strings. The ID should be retained for identification, but not treated as a numeric feature.


In [42]:
print('ID data type in train dataset :',train_df[ID_COL].dtype)
print(train_df[ID_COL].map(type).value_counts())
print('_'*25)
print('ID data type in test dataset :',test_df[ID_COL].dtype)
print(test_df[ID_COL].map(type).value_counts())

ID data type in train dataset : object
Tour_ID
<class 'str'>    18478
Name: count, dtype: int64
_________________________
ID data type in test dataset : object
Tour_ID
<class 'str'>    6169
Name: count, dtype: int64


All values in `Tour_ID` are strings in both datasets. The identifier format is consistent and should not be converted into a numeric variable.


### Verify Target Availability

Confirm that `cost_category` exists in train and is absent from test. This is the expected structure for a supervised learning task with an unlabeled test set.


In [43]:
print("TARGET in train:", "cost_category" in train_df.columns)
print("TARGET in test:", "cost_category" in test_df.columns)

TARGET in train: True
TARGET in test: False


The target is present in train and absent from test, which is the expected structure for this supervised classification task.


### Inspect the Target Data Type

Check that the target values are consistently represented as strings before encoding or model training.


In [44]:
print('TARGET data type in train dataset :',train_df[TARGET_COL].dtype)
print(train_df[TARGET_COL].map(type).value_counts())

TARGET data type in train dataset : object
cost_category
<class 'str'>    18478
Name: count, dtype: int64


### Review Target Classes and Their Frequencies

Inspect the class labels and their counts. The distribution is not balanced: `Highest Cost` is the smallest class, so later model evaluation should use metrics that account for class imbalance.


In [45]:
display(TARGET_CLASSES)
train_df[TARGET_COL].value_counts()

['Lower Cost',
 'Low Cost',
 'Normal Cost',
 'High Cost',
 'Higher Cost',
 'Highest Cost']

cost_category
Normal Cost     5465
Higher Cost     4858
High Cost       3672
Lower Cost      2563
Low Cost        1564
Highest Cost     356
Name: count, dtype: int64

The target contains six classes with different frequencies. `Highest Cost` is the minority class, so use stratified splitting and report a class-sensitive metric such as macro F1 alongside other evaluation metrics.


## 7. Handle Invalid and Inconsistent Values

Review numerical ranges and categorical values for spelling errors, duplicate category labels, and inconsistent naming. Apply each correction consistently to both datasets so that the model sees the same category definitions during training and prediction.


### Recheck Numerical Ranges Before Standardization

Repeat the numerical summary after the earlier cleaning steps. Use it as a reference while reviewing categorical inconsistencies and potential invalid values.


In [47]:
display(train_df[BASE_NUMERIC_COLS].describe().T)
display(test_df[BASE_NUMERIC_COLS].describe().T)

,count,mean,std,min,25%,50%,75%,max
total_female,18478.0,0.932839,1.204878,0.0,0.0,1.0,1.0,49.0
total_male,18478.0,0.997727,1.172420,0.0,1.0,1.0,1.0,58.0
night_mainland,18478.0,9.140491,14.132483,0.0,3.0,6.0,11.0,365.0
night_zanzibar,18478.0,2.495671,5.278308,0.0,0.0,0.0,4.0,240.0


,count,mean,std,min,25%,50%,75%,max
total_female,6167.0,0.922491,1.173067,0.0,0.0,1.0,1.0,30.0
total_male,6168.0,1.017510,1.526274,0.0,1.0,1.0,1.0,90.0
night_mainland,6169.0,9.311720,16.366900,0.0,3.0,6.0,11.0,664.0
night_zanzibar,6169.0,2.585832,5.465058,0.0,0.0,0.0,4.0,174.0


### Review Training Categorical Values

Print the unique values in each categorical feature to identify spelling variations, inconsistent labels, and categories that may need standardization.


In [49]:
for col in train_df[BASE_CATEGORICAL_COLS]:
    print(train_df[col].name,'\n', train_df[col].unique())
    print(train_df[col].dtype, '\n')

country 
 ['ITALY' 'UNITED KINGDOM' 'UNITED STATES OF AMERICA' 'RWANDA' 'FRANCE'
 'SWIZERLAND' 'NIGERIA' 'AUSTRALIA' 'SPAIN' 'DENMARK' 'BELGIUM' 'ZAMBIA'
 'NETHERLANDS' 'CANADA' 'INDIA' 'NORWAY' 'ZIMBABWE' 'MORROCO' 'CHINA'
 'GERMANY' 'SOUTH AFRICA' 'JAPAN' 'UAE' 'RUSSIA' 'KENYA' 'GREECE' 'POLAND'
 'AUSTRIA' 'MALAYSIA' 'CZECH REPUBLIC' 'KOREA' 'DRC' 'CYPRUS' 'KUWAIT'
 'BRAZIL' 'CHILE' 'BULGARIA' 'UNITED ARAB EMIRATES' 'ISRAEL' 'NEW ZEALAND'
 'UGANDA' 'IRELAND' 'SWEDEN' 'QATAR' 'CONGO' 'BURUNDI' 'FINLAND' 'GHANA'
 'OMAN' 'LEBANON' 'LUXEMBOURG' 'IVORY COAST' 'NAMIBIA' 'ROMANIA'
 'THAILAND' 'SUDAN' 'SINGAPORE' 'PORTUGAL' 'MALAWI' 'MALT' 'EGYPT'
 'PAKISTAN' 'TANZANIA' 'ARGENTINA' 'SRI LANKA' 'BOTSWANA' 'VIETNAM'
 'BERMUDA' 'TAIWAN' 'SENEGAL' 'AFGHANISTAN' 'COLOMBIA' 'JAMAICA' 'BAHRAIN'
 'COSTARICA' 'COMORO' 'URUGUAY' 'SAUD ARABIA' 'ETHIOPIA' 'PHILIPINES'
 'ARMENIA' 'SCOTLAND' 'MEXICO' 'MADAGASCAR' 'SERBIA' 'SLOVAKIA' 'IRAN'
 'SWAZILAND' 'PERU' 'LITHUANIA' 'ALGERIA' 'SEYCHELLES' 'MOZAMBIQUE

### Review Test Categorical Values

Inspect the test categories using the same approach. This also helps identify categories that appear in test but were not observed in train.


In [50]:
for col in test_df[BASE_CATEGORICAL_COLS]:
    print(test_df[col].name,'\n', test_df[col].unique())
    print(test_df[col].dtype, '\n')

country 
 ['KOREA' 'UNITED KINGDOM' 'ITALY' 'KENYA' 'ZAMBIA' 'UGANDA' 'NETHERLANDS'
 'OMAN' 'ZIMBABWE' 'BELGIUM' 'DRC' 'UNITED STATES OF AMERICA' 'FRANCE'
 'CANADA' 'CZECH REPUBLIC' 'UAE' 'SPAIN' 'SOUTH AFRICA' 'SWEDEN' 'NORWAY'
 'CHINA' 'GERMANY' 'RWANDA' 'INDIA' 'AUSTRIA' 'DJIBOUT' 'IRELAND'
 'SWIZERLAND' 'AUSTRALIA' 'PHILIPINES' 'BURUNDI' 'JAPAN' 'THAILAND'
 'PORTUGAL' 'DENMARK' 'MOZAMBIQUE' 'UNITED ARAB EMIRATES' 'TAIWAN'
 'BRAZIL' 'HUNGARY' 'SRI LANKA' 'TUNISIA' 'INDONESIA' 'ISRAEL'
 'NEW ZEALAND' 'POLAND' 'CONGO' 'ESTONIA' 'COLOMBIA' 'MALAYSIA' 'TANZANIA'
 'BULGARIA' 'RUSSIA' 'COMORO' 'UKRAIN' 'LEBANON' 'URUGUAY' 'SOMALI'
 'FINLAND' 'ETHIOPIA' 'SCOTLAND' 'TURKEY' 'SLOVENIA' 'MORROCO' 'SWAZILAND'
 'LUXEMBOURG' 'MALAWI' 'SINGAPORE' 'SUDAN' 'PAPUA NEW GUINEA' 'PAKISTAN'
 'EGYPT' 'CYPRUS' 'QATAR' 'CAMEROON' 'ANGOLA' 'SAUD ARABIA' 'GREECE'
 'KAZAKHSTAN' 'MADAGASCAR' 'CAMBODIA' 'BENIN' 'ARGENTINA' 'VIETNAM' 'PERU'
 'COSTARICA' 'ROMANIA' 'MEXICO' 'DOMINICA' 'GHANA' 'BOSNIA' 'ECUADO'
 'B

The `country` feature contains inconsistent labels, including alternative names for the same country and spelling mistakes. These values should be standardized before encoding. The mapping below is applied to both datasets.

The decision to combine `SCOTLAND` with `UNITED KINGDOM` is a geographic standardization choice; keep it only if that level of grouping matches the project's definition of country of origin.


### Standardize Country Labels

Apply the same country-name mapping to the training and test datasets. The mapping consolidates alternate labels and corrects spelling mistakes while keeping category values consistent across both sets.


In [51]:
train_df['country'] = train_df['country'].replace({
    'UNITED ARAB EMIRATES':'UAE', # is same country
    'BURGARIA' : 'BULGARIA', # is same country
    
    'SCOTLAND': 'UNITED KINGDOM', # scotland it's in united kingdom 

    # Spelling mistakes    
    'SWIZERLAND': 'SWITZERLAND',
    'DJIBOUT': 'DJIBOUTI',
    'PHILIPINES': 'PHILIPPINES',
    'UKRAIN': 'UKRAINE',
    'SOMALI': 'SOMALIA',
    'MORROCO': 'MOROCCO',
    'SAUD ARABIA': 'SAUDI ARABIA',
    'COSTARICA': 'COSTA RICA',
    'COMORO': 'COMOROS',
    'ECUADO': 'ECUADOR',
    'MALT': 'MALTA',
    'TRINIDAD TOBACCO': 'TRINIDAD AND TOBAGO' # only in train data

})
test_df['country'] = test_df['country'].replace({
    'UNITED ARAB EMIRATES':'UAE', # is same country
    'BURGARIA' : 'BULGARIA', # is same country
    
    'SCOTLAND': 'UNITED KINGDOM', # scotland it's in united kingdom 

    # Spelling mistakes
    'SWIZERLAND': 'SWITZERLAND',
    'DJIBOUT': 'DJIBOUTI',
    'PHILIPINES': 'PHILIPPINES',
    'UKRAIN': 'UKRAINE',
    'SOMALI': 'SOMALIA',
    'MORROCO': 'MOROCCO',
    'SAUD ARABIA': 'SAUDI ARABIA',
    'COSTARICA': 'COSTA RICA',
    'COMORO': 'COMOROS',
    'ECUADO': 'ECUADOR',
    'MALT': 'MALTA'
})

/tmp/ipykernel_37971/2709164182.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['country'] = train_df['country'].replace({


The mapping standardizes the listed country labels in both datasets. After applying it, review the remaining unique values to make sure no important spelling variants were missed.


### Standardize the `main_activity` Labels

Check whether similar category labels represent the same activity but differ because of a spelling mistake.


### Inspect `main_activity` Frequencies

Count the training labels to identify categories that appear to represent the same activity but differ in spelling.


In [52]:
train_df['main_activity'].value_counts()

main_activity
Widlife Tourism             5988
Beach Tourism               4170
Wildlife Tourism            2264
Conference Tourism          1917
Hunting Tourism             1710
Cultural Tourism            1443
Mountain Climbing            528
Business                     280
Bird Tourism                 144
Diving and Sport Fishing      34
Name: count, dtype: int64

The training data contains two labels for the same activity: `Widlife Tourism` and `Wildlife Tourism`. Since the first label is a spelling mistake, both should be mapped to `Wildlife Tourism`.


### Apply the Activity-Name Correction

Replace the misspelled `Widlife Tourism` label with `Wildlife Tourism` in both datasets. This prevents one activity from being treated as two separate categories.


In [53]:
train_df['main_activity'] = train_df['main_activity'].replace({'Widlife Tourism':'Wildlife Tourism'})
test_df['main_activity'] = test_df['main_activity'].replace({'Widlife Tourism':'Wildlife Tourism'})

/tmp/ipykernel_37971/1983061618.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['main_activity'] = train_df['main_activity'].replace({'Widlife Tourism':'Wildlife Tourism'})


The correction combines the two spelling variants into one category. This reduces artificial category fragmentation without changing the meaning of the feature.


### Verify Corrected Training Labels

Check the training frequency table again to confirm that the two labels have been consolidated.


In [54]:
train_df['main_activity'].value_counts()

main_activity
Wildlife Tourism            8252
Beach Tourism               4170
Conference Tourism          1917
Hunting Tourism             1710
Cultural Tourism            1443
Mountain Climbing            528
Business                     280
Bird Tourism                 144
Diving and Sport Fishing      34
Name: count, dtype: int64

### Verify Corrected Test Labels

Check the test frequency table to confirm that the same correction was applied consistently.


In [55]:
test_df['main_activity'].value_counts()

main_activity
Wildlife Tourism            2779
Beach Tourism               1432
Conference Tourism           612
Hunting Tourism              523
Cultural Tourism             495
Mountain Climbing            198
Business                      90
Bird Tourism                  34
Diving and Sport Fishing       6
Name: count, dtype: int64

The test distribution also contains the standardized label `Wildlife Tourism`, confirming that the replacement was applied to test.


The categorical spelling corrections have been applied to both datasets. The next section reviews extreme numerical observations; these should be evaluated separately from categorical cleaning.


## 8. Investigate Outliers

Outlier detection flags unusual observations; it does not automatically mean those observations are errors. Review the size and duration features using percentiles, then decide whether extreme values are valid based on domain knowledge.


### Review High Percentiles and Extreme Values

Compare the 90th, 95th, 99th, and 99.5th percentiles with the maximum values. Large gaps indicate right-skewed distributions and extreme observations that need domain review.


In [57]:
train_df[BASE_NUMERIC_COLS].describe(percentiles=[0.90, 0.95, 0.99, 0.995]).T

,count,mean,std,min,50%,90%,95%,99%,99.5%,max
total_female,18478.0,0.932839,1.204878,0.0,1.0,2.0,3.0,5.0,6.0,49.0
total_male,18478.0,0.997727,1.172420,0.0,1.0,2.0,2.0,5.0,6.0,58.0
night_mainland,18478.0,9.140491,14.132483,0.0,6.0,17.0,28.0,61.0,90.0,365.0
night_zanzibar,18478.0,2.495671,5.278308,0.0,0.0,7.0,10.0,17.0,22.0,240.0


The percentile tables show large gaps between upper percentiles and maximum values, especially for `night_mainland` and `night_zanzibar`. Do not remove or cap these observations based only on their size; first verify whether they are possible travel durations and decide on a documented, train-fitted treatment if needed.


## 9. Validate Preprocessing Decisions

Run a final set of checks to confirm the processed data shape, missing-value status, duplicate status, data types, categorical values, infinite values, and target integrity. Any remaining issues must be documented and handled before model training or prediction.


### Validate Dataset Dimensions

Confirm the final row and column counts. The training set has 18,478 rows and 21 columns after the current missing-row removal; the test set retains all 6,169 rows and 20 columns.


In [58]:
print(f'shape of train dataset : {train_df.shape}')
print(f'rows in train dataset : {len(train_df)}')
print(f'columns in train dataset : {train_df.shape[1]}')
print("_"*50,'\n')
print(f'shape of test dataset : {test_df.shape}')
print(f'rows in test dataset : {len(test_df)}')
print(f'columns in test dataset : {test_df.shape[1]}')


shape of train dataset : (18478, 21)
rows in train dataset : 18478
columns in train dataset : 21
__________________________________________________ 

shape of test dataset : (6169, 20)
rows in test dataset : 6169
columns in test dataset : 20


The expected column counts are preserved: 21 columns in train and 20 in test. The training row count reflects the removal of 28 incomplete rows, while the test row count remains unchanged.


### Validate Remaining Missing Values

The training dataset has no missing values after row removal. The test dataset still contains missing values in `travel_with`, `total_female`, and `total_male`; these must be handled before prediction using rules or imputers defined from training data.


In [59]:
fmva(train_df, test_df)

Missing Values in train dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,,
total_female,,
total_male,,
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


__________________________________________________ 

Missing Values in test dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,3.0,0.05
total_female,2.0,0.03
total_male,1.0,0.02
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


The training dataset is complete, but test still has **3 missing `travel_with` values, 2 missing `total_female` values, and 1 missing `total_male` value**. This is an outstanding preprocessing task, not a validation pass.


**Test-set handling:** The test dataset is inspected and receives the same deterministic cleaning rules as the training dataset. It must retain all original rows. Any remaining missing values should be handled with an imputation strategy fitted on the training data and then applied to the test data; do not calculate imputation statistics from the test set.


### Validate Full-Row Duplicates

Confirm that no exact duplicate rows remain in either dataset after preprocessing.


In [60]:
print(f'duplicated row in train dataset is {train_df.duplicated().sum()}')
print(f'duplicated row in test dataset is {test_df.duplicated().sum()}')


duplicated row in train dataset is 0
duplicated row in test dataset is 0


### Validate Unique Identifiers

Check that `Tour_ID` has no duplicate values within either dataset. This is a separate check from full-row duplication.


In [61]:
print(f'duplicated ID column in train dataset is {train_df[ID_COL].duplicated().sum()}')
print(f'duplicated ID column in test dataset is {test_df[ID_COL].duplicated().sum()}')


duplicated ID column in train dataset is 0
duplicated ID column in test dataset is 0


### Recheck Data-Type Consistency

Compare the shared feature types one final time. The target column should appear only in train, while all predictor columns should have compatible types.


In [62]:
data_values_com = pd.DataFrame({
    'train types' : train_df.dtypes,
    'test types' : test_df.dtypes})
data_values_com

,train types,test types
Tour_ID,object,object
age_group,object,object
cost_category,object,NaN
country,object,object
first_trip_tz,object,object
info_source,object,object
main_activity,object,object
night_mainland,int64,int64
night_zanzibar,int64,int64
package_accomodation,object,object


All shared features retain compatible data types. The missing target column in test is intentional and should not be treated as a schema error.


### Review Final Training Categories

Inspect the final training category values after standardization. These values will define the categories observed during model fitting.


In [64]:
for col in train_df[BASE_CATEGORICAL_COLS]:
    print(train_df[col].name,'\n', train_df[col].unique())
    print(train_df[col].dtype, '\n')

country 
 ['ITALY' 'UNITED KINGDOM' 'UNITED STATES OF AMERICA' 'RWANDA' 'FRANCE'
 'SWITZERLAND' 'NIGERIA' 'AUSTRALIA' 'SPAIN' 'DENMARK' 'BELGIUM' 'ZAMBIA'
 'NETHERLANDS' 'CANADA' 'INDIA' 'NORWAY' 'ZIMBABWE' 'MOROCCO' 'CHINA'
 'GERMANY' 'SOUTH AFRICA' 'JAPAN' 'UAE' 'RUSSIA' 'KENYA' 'GREECE' 'POLAND'
 'AUSTRIA' 'MALAYSIA' 'CZECH REPUBLIC' 'KOREA' 'DRC' 'CYPRUS' 'KUWAIT'
 'BRAZIL' 'CHILE' 'BULGARIA' 'ISRAEL' 'NEW ZEALAND' 'UGANDA' 'IRELAND'
 'SWEDEN' 'QATAR' 'CONGO' 'BURUNDI' 'FINLAND' 'GHANA' 'OMAN' 'LEBANON'
 'LUXEMBOURG' 'IVORY COAST' 'NAMIBIA' 'ROMANIA' 'THAILAND' 'SUDAN'
 'SINGAPORE' 'PORTUGAL' 'MALAWI' 'MALTA' 'EGYPT' 'PAKISTAN' 'TANZANIA'
 'ARGENTINA' 'SRI LANKA' 'BOTSWANA' 'VIETNAM' 'BERMUDA' 'TAIWAN' 'SENEGAL'
 'AFGHANISTAN' 'COLOMBIA' 'JAMAICA' 'BAHRAIN' 'COSTA RICA' 'COMOROS'
 'URUGUAY' 'SAUDI ARABIA' 'ETHIOPIA' 'PHILIPPINES' 'ARMENIA' 'MEXICO'
 'MADAGASCAR' 'SERBIA' 'SLOVAKIA' 'IRAN' 'SWAZILAND' 'PERU' 'LITHUANIA'
 'ALGERIA' 'SEYCHELLES' 'MOZAMBIQUE' 'UKRAINE' 'NEPAL' 'CROATIA

### Review Final Test Categories

Inspect final test categories and compare them with the training vocabulary. The later encoding pipeline should safely handle categories that appear in test but were not seen during training.


In [66]:
for col in test_df[BASE_CATEGORICAL_COLS]:
    print(test_df[col].name,'\n', test_df[col].unique())
    print(test_df[col].dtype, '\n')

country 
 ['KOREA' 'UNITED KINGDOM' 'ITALY' 'KENYA' 'ZAMBIA' 'UGANDA' 'NETHERLANDS'
 'OMAN' 'ZIMBABWE' 'BELGIUM' 'DRC' 'UNITED STATES OF AMERICA' 'FRANCE'
 'CANADA' 'CZECH REPUBLIC' 'UAE' 'SPAIN' 'SOUTH AFRICA' 'SWEDEN' 'NORWAY'
 'CHINA' 'GERMANY' 'RWANDA' 'INDIA' 'AUSTRIA' 'DJIBOUTI' 'IRELAND'
 'SWITZERLAND' 'AUSTRALIA' 'PHILIPPINES' 'BURUNDI' 'JAPAN' 'THAILAND'
 'PORTUGAL' 'DENMARK' 'MOZAMBIQUE' 'TAIWAN' 'BRAZIL' 'HUNGARY' 'SRI LANKA'
 'TUNISIA' 'INDONESIA' 'ISRAEL' 'NEW ZEALAND' 'POLAND' 'CONGO' 'ESTONIA'
 'COLOMBIA' 'MALAYSIA' 'TANZANIA' 'BULGARIA' 'RUSSIA' 'COMOROS' 'UKRAINE'
 'LEBANON' 'URUGUAY' 'SOMALIA' 'FINLAND' 'ETHIOPIA' 'TURKEY' 'SLOVENIA'
 'MOROCCO' 'SWAZILAND' 'LUXEMBOURG' 'MALAWI' 'SINGAPORE' 'SUDAN'
 'PAPUA NEW GUINEA' 'PAKISTAN' 'EGYPT' 'CYPRUS' 'QATAR' 'CAMEROON'
 'ANGOLA' 'SAUDI ARABIA' 'GREECE' 'KAZAKHSTAN' 'MADAGASCAR' 'CAMBODIA'
 'BENIN' 'ARGENTINA' 'VIETNAM' 'PERU' 'COSTA RICA' 'ROMANIA' 'MEXICO'
 'DOMINICA' 'GHANA' 'BOSNIA' 'ECUADOR' 'BERMUDA' 'NAMIBIA' 'SLOVAKI

The category lists can reveal new test categories, but they do not automatically prove that every test category exists in train. The encoding stage should use an approach that supports unknown categories (for example, an encoder with `handle_unknown='ignore'`) or explicitly validate the category vocabularies.


### Create an Infinite-Value Check

Define a reusable function that checks numerical columns for positive or negative infinity and displays their descriptive statistics. Infinite values should be handled before model fitting.


In [69]:
def inf(df):
    infinite_values = np.isinf(df[BASE_NUMERIC_COLS]).sum()
    print(f'Infinite values : \n{infinite_values}')
    display(df[BASE_NUMERIC_COLS].describe().T)


### Check Infinite Values in Training

Run the check on train. The reported infinite-value counts are zero for all numerical features.


In [70]:
inf(train_df)

Infinite values : 
total_female      0
total_male        0
night_mainland    0
night_zanzibar    0
dtype: int64


,count,mean,std,min,25%,50%,75%,max
total_female,18478.0,0.932839,1.204878,0.0,0.0,1.0,1.0,49.0
total_male,18478.0,0.997727,1.172420,0.0,1.0,1.0,1.0,58.0
night_mainland,18478.0,9.140491,14.132483,0.0,3.0,6.0,11.0,365.0
night_zanzibar,18478.0,2.495671,5.278308,0.0,0.0,0.0,4.0,240.0


No infinite values are reported in the training numerical columns. The descriptive statistics also confirm the right-skewed ranges observed earlier.


### Check Infinite Values in Test

Run the same check on test. The reported infinite-value counts are zero for all numerical features.


In [71]:
inf(test_df)

Infinite values : 
total_female      0
total_male        0
night_mainland    0
night_zanzibar    0
dtype: int64


,count,mean,std,min,25%,50%,75%,max
total_female,6167.0,0.922491,1.173067,0.0,0.0,1.0,1.0,30.0
total_male,6168.0,1.017510,1.526274,0.0,1.0,1.0,1.0,90.0
night_mainland,6169.0,9.311720,16.366900,0.0,3.0,6.0,11.0,664.0
night_zanzibar,6169.0,2.585832,5.465058,0.0,0.0,0.0,4.0,174.0


No infinite values are reported in the test numerical columns. The same numerical validation has therefore passed for both datasets.


### Validate the Target Column

Confirm that the training target contains no missing values and review the final class frequencies. This is an important check before creating training and validation splits.


In [72]:
print(f'missing values in TARGET column : {train_df[TARGET_COL].isnull().sum()}')
print(f'\nTARGET class : \n{train_df[TARGET_COL].value_counts()}')


missing values in TARGET column : 0

TARGET class : 
cost_category
Normal Cost     5465
Higher Cost     4858
High Cost       3672
Lower Cost      2563
Low Cost        1564
Highest Cost     356
Name: count, dtype: int64


The training target has **zero missing values**. The class distribution remains imbalanced, with `Highest Cost` as the smallest class. Consider this when selecting validation metrics and interpreting model performance.


## 10. Preprocessing Summary

Summarize the main data-quality findings, the transformations applied, and the issues that remain for the next stage.


### Summary of the Current Preprocessing Stage

**Completed checks and transformations**

- Loaded the raw training and test datasets and reviewed their structure.
- Investigated missing values and applied rule-based inference to selected `travel_with` records.
- Removed 28 incomplete rows from train using `dropna()`; the training set now contains 18,478 rows.
- Checked duplicate rows and ID uniqueness; no duplicates were found.
- Reviewed data types and confirmed that `Tour_ID` is stored as text.
- Standardized selected country labels and corrected the `Widlife Tourism` spelling.
- Checked numerical ranges and confirmed that no infinite values were found.

**Outstanding items before modeling**

- Test still has 6 missing numerical/category values across `travel_with`, `total_female`, and `total_male`. Handle them with a strategy fitted on training data, without dropping test rows.
- Review the decision to drop incomplete training rows and compare it with train-fitted imputation.
- Decide whether extreme numerical values are valid and document any outlier treatment.
- Validate unseen test categories and configure the encoding pipeline to handle them safely.
- Fix or remove the diagnostic cells that use the undefined variables `omale_and_ofemale` and `df`.

The processed datasets should be saved only after the remaining decisions are implemented and the full preprocessing pipeline is validated.
